# LoRAForge - Fine-tune, merge, and quantize to GGUF

Runs on a **free T4 GPU** (Kaggle preferred). End to end this notebook:
1. loads **Qwen3 4B Instruct** in 4-bit (QLoRA via Unsloth),
2. fine-tunes a LoRA adapter on the **Bitext customer-support** dataset,
3. merges the adapter into the base model,
4. converts + quantizes to a single **Q4_K_M `.gguf`**,
5. saves `loss.png`.

## Before you Run All
- Settings -> Accelerator -> **GPU T4** (confirm a GPU shows under `!nvidia-smi`).
- No Hugging Face token needed: the base model and dataset are both ungated.
- Leave `SMOKE = True` for a fast ~60-step sanity run; set `SMOKE = False` for the real 3-epoch train.

When it finishes, download from the output panel: **`loraforge-qwen3-4b-q4_k_m.gguf`** and **`loss.png`**. Drop the `.gguf` into `loraforge/models/` and tell Claude the path.


## 0. GPU check + install


In [ ]:
!nvidia-smi

In [ ]:
# Unsloth pulls compatible torch/trl/peft/bitsandbytes. ~3-5 min on Kaggle.
%pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q matplotlib

## 1. Hugging Face login (optional - the Qwen3 base is ungated)


In [ ]:
# The Qwen3 base model (unsloth/Qwen3-4B-Instruct-2507) and the Bitext dataset are BOTH
# ungated, so no token is required for a headless run. If you ever point BASE_MODEL at a
# gated repo, set HF_TOKEN (env var) or add a Kaggle secret named HF_TOKEN.
import os
token = os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if token:
    from huggingface_hub import login
    login(token)
    print('HF login OK')
else:
    print('No HF token found - continuing unauthenticated (base model + dataset are ungated).')


## 2. Config
Edit here. `SMOKE = True` runs ~60 steps to prove the pipeline; flip to `False` for the real run.


In [ ]:
SMOKE = False            # set False for the full 3-epoch fine-tune

BASE_MODEL   = 'unsloth/Qwen3-4B-Instruct-2507'  # Qwen3 4B instruct (non-thinking), ungated
OUTPUT_NAME  = 'loraforge-qwen3-4b'        # base name for the .gguf
MAX_SEQ_LEN  = 2048      # cap to avoid T4 OOM
LORA_R       = 16
LORA_ALPHA   = 32
LR           = 2e-4
EPOCHS       = 3
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
SEED         = 42
QUANT        = 'q4_k_m'  # size/quality sweet spot

DATASET_ID   = 'bitext/Bitext-customer-support-llm-chatbot-training-dataset'
MAX_STEPS    = 60 if SMOKE else -1
SMOKE_ROWS   = 500       # subsample the dataset in smoke mode for speed

## 3. Load base model in 4-bit + attach LoRA


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
    dtype = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0.0,
    bias = 'none',
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth',
    random_state = SEED,
)

## 4. Load + format the dataset
We download Bitext from HF and map it to the model's **native chat template** via
`tokenizer.apply_chat_template`. A 10% slice is held out and never trained on (eval
contamination guard). No ChatML hardcoding: the template matches whatever `BASE_MODEL` is.


In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer)   # use the tokenizer's built-in template

raw = load_dataset(DATASET_ID, split='train')
if SMOKE:
    raw = raw.shuffle(seed=SEED).select(range(min(SMOKE_ROWS, len(raw))))

def to_text(batch):
    texts = []
    for instr, resp in zip(batch['instruction'], batch['response']):
        msgs = [{'role':'user','content':instr.strip()},
                {'role':'assistant','content':resp.strip()}]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {'text': texts}

cols = raw.column_names
split = raw.train_test_split(test_size=0.1, seed=SEED)
train_ds = split['train'].map(to_text, batched=True, remove_columns=cols)
test_ds  = split['test'].map(to_text,  batched=True, remove_columns=cols)
print(f'train={len(train_ds)}  held-out test={len(test_ds)}')
print(train_ds[0]['text'][:400])

## 5. Train


In [ ]:
from trl import SFTConfig, SFTTrainer

cfg = SFTConfig(
    output_dir = 'checkpoints',
    save_strategy = 'no',   # no intermediate checkpoints (saves disk on the long run)
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_steps = 5,
    num_train_epochs = (1 if MAX_STEPS > 0 else EPOCHS),
    max_steps = MAX_STEPS,
    learning_rate = LR,
    logging_steps = 1,
    optim = 'adamw_8bit',
    weight_decay = 0.01,
    lr_scheduler_type = 'linear',
    seed = SEED,
    report_to = 'none',
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LEN,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, args=cfg)
stats = trainer.train()
print('final loss:', stats.training_loss)

## 6. Loss curve -> loss.png


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

hist = trainer.state.log_history
steps  = [h['step'] for h in hist if 'loss' in h]
losses = [h['loss'] for h in hist if 'loss' in h]
plt.figure(figsize=(8,5))
plt.plot(steps, losses, label='train loss')
plt.xlabel('step'); plt.ylabel('loss'); plt.title('LoRAForge training loss'); plt.legend()
plt.tight_layout(); plt.savefig('loss.png', dpi=120)
print('saved loss.png'); plt.show()

## 7. Quick sanity generation
Make sure the tuned model answers a support question coherently before exporting.


In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{'role':'user','content':'How long do refunds take?'}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 8. Merge + export to GGUF (Q4_K_M)
Unsloth merges the adapter into fp16 and runs the llama.cpp conversion + quantization in
one call. The tokenizer (with added tokens) is saved alongside, so the GGUF vocab stays
aligned. Output is a single `.gguf` under `gguf_model/`.


In [ ]:
model.save_pretrained_gguf('gguf_model', tokenizer, quantization_method=QUANT)

import glob, os, shutil
# Unsloth writes the merged fp16 to gguf_model/ and the GGUF(s) to gguf_model_gguf/.
# Search recursively and prefer the quantized (QUANT) file over the larger f16.
gguf_files = glob.glob('**/*.gguf', recursive=True)
assert gguf_files, f'no .gguf produced; cwd contents: {os.listdir(".")}'
qkey = QUANT.replace('_','').lower()
pref = [f for f in gguf_files if qkey in os.path.basename(f).replace('_','').lower()]
src = pref[0] if pref else sorted(gguf_files, key=os.path.getsize)[0]
dst = f'{OUTPUT_NAME}-{QUANT}.gguf'
shutil.copy(src, dst)
print('GGUF ready:', dst, f'({os.path.getsize(dst)/1e9:.2f} GB)  [from {src}]')

# free bulky intermediates so the Kaggle output download stays small
for d in ('gguf_model','gguf_model_gguf'):
    shutil.rmtree(d, ignore_errors=True)
print('cleaned intermediate dirs')


## 9. Download
From the Kaggle **Output** panel (or the file browser), download:
- **`loraforge-qwen3-4b-q4_k_m.gguf`** -> put it in `loraforge/models/`
- **`loss.png`** -> put it in `loraforge/` (the README references it)

Then tell Claude the model path and it takes over: `ollama create`, the FastAPI proxy,
the EvalKit base-vs-tuned run, and the docs.


In [ ]:
import os
for f in ['%s-%s.gguf' % (OUTPUT_NAME, QUANT), 'loss.png']:
    print(f, '->', f'{os.path.getsize(f)/1e6:.1f} MB' if os.path.exists(f) else 'MISSING')